In [ ]:

import os, random, pickle, itertools, zipfile
from datetime import datetime
import numpy as np

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ================== 1) PyTorch & TorchVision ==================
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from typing import Dict, List
from PIL import Image

print(f"[INFO] Started at {datetime.now().isoformat()}")

# ================== 2) CUDA & Seeds ==================
print("[INFO] torch.cuda.is_available() ->", torch.cuda.is_available())
if torch.cuda.is_available():
    try:
        _x = torch.zeros(1, device='cuda'); print("[INFO] CUDA sanity OK on device:", _x.device)
    except Exception as e:
        raise SystemExit("[FATAL] CUDA sanity failed: " + repr(e))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("[INFO] Using device:", device)
if device.type == 'cuda':
    print("[INFO] CUDA device:", torch.cuda.get_device_name(0))
    print("[INFO] CUDA capability:", torch.cuda.get_device_capability(0))
    print("[INFO] CUDA current mem (MB):", torch.cuda.memory_reserved() / (1024**2))

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if device.type == 'cuda': torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"[INFO] Seeds set to {SEED}")

# ================== 3) paths ==================
model_path      = "/content/drive/MyDrive/ML_Project/project_files/Group_norm/Gtask8_best_test_for_finetune_ResNet18_GN_CS.pth"

topk_path       = "/content/drive/MyDrive/ML_Project/project_files/Group_norm/groupNorm_CS_task1_2_3_4_5_6_7_8_tiny_imageNet_topk.pkl"
neighbors_path  = "/content/drive/MyDrive/ML_Project/project_files/Group_norm/groupNorm_CS_task1_2_3_4_5_6_7_8_tiny_imageNet_neighbors.pkl"

ckpt_dir        = "/content/drive/MyDrive/ML_Project/project_files/Group_norm"
os.makedirs(ckpt_dir, exist_ok=True)

def _check_file(path, tag):
    if not os.path.exists(path): raise FileNotFoundError(f"[MISSING] {tag}: {path}")
    print(f"[OK] {tag} exists ({os.path.getsize(path)/(1024**2):.2f} MB): {path}")

_check_file(model_path, "Checkpoint(prev)")
_check_file(topk_path, "CS Top-K")
_check_file(neighbors_path, "CS Neighbors")

# ================== 4) Tiny-ImageNet (processed .npy) ==================
BASE      = "/content/drive/MyDrive/ML_Project/project_files/Group_norm"
ZIP_PATH  = f"{BASE}/tiny-imagenet-processed.zip"
DATA_ROOT = "/content/drive/MyDrive/ML_Project/data/TINYIMG"

def ensure_extracted(zip_path: str, data_root: str) -> str:
    processed_dir = os.path.join(data_root, "processed")
    if os.path.isdir(processed_dir) and len(os.listdir(processed_dir)) > 0:
        print(f"[INFO] Found processed data at: {processed_dir}")
        return data_root
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f"ZIP not found at:\n{zip_path}")
    print(f"[INFO] Extracting ZIP from:\n{zip_path}\n-> to:\n{data_root}")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(data_root)
    assert os.path.isdir(processed_dir), "processed/ folder missing after unzip!"
    return data_root

ensure_extracted(ZIP_PATH, DATA_ROOT)

TIN_IMAGENET_MEAN = (0.4802, 0.4480, 0.3975)
TIN_IMAGENET_STD  = (0.2770, 0.2691, 0.2821)

class TinyImagenet(Dataset):
    """
    expects processed/
      x_train_01.npy ... x_train_20.npy
      y_train_01.npy ... y_train_20.npy
      x_val_01.npy   ... x_val_20.npy
      y_val_01.npy   ... y_val_20.npy
    """
    def __init__(self, root: str, train: bool=True, transform=None):
        self.root = root; self.train = train; self.transform = transform
        split = "train" if self.train else "val"
        xs, ys = [], []
        for num in range(20):
            xs.append(np.load(os.path.join(root, f'processed/x_{split}_{num+1:02d}.npy')));
            ys.append(np.load(os.path.join(root, f'processed/y_{split}_{num+1:02d}.npy')))
        self.data = np.concatenate(np.array(xs))
        self.targets = np.concatenate(np.array(ys)).astype(int)

    def __len__(self): return len(self.data)

    def _to_uint8_img(self, arr):
        if arr.dtype != np.uint8:
            arr = (np.clip(arr, 0.0, 1.0) * 255.0).astype(np.uint8)
        return Image.fromarray(arr)

    def __getitem__(self, index):
        img, target = self.data[index], int(self.targets[index])
        if img.ndim == 3 and img.shape[0] == 3 and (img.shape[-1] != 3):
            img = np.transpose(img, (1, 2, 0))
        if img.ndim == 2: img = np.stack([img, img, img], axis=-1)
        if img.ndim == 3 and img.shape[-1] == 1: img = np.repeat(img, 3, axis=-1)
        img = self._to_uint8_img(img)
        if self.transform is not None: img = self.transform(img)
        return img, target

tf_train = transforms.Compose([
    transforms.RandomCrop(64, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(TIN_IMAGENET_MEAN, TIN_IMAGENET_STD),
])
tf_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(TIN_IMAGENET_MEAN, TIN_IMAGENET_STD),
])

print("[INFO] Loading Tiny-ImageNet ...")
train_set = TinyImagenet(root=DATA_ROOT, train=True,  transform=tf_train)
test_set  = TinyImagenet(root=DATA_ROOT, train=False, transform=tf_test)
print(f"[INFO] Train size={len(train_set)} | Val/Test size={len(test_set)}")

def build_tasks(num_classes: int = 200, classes_per_task: int = 20) -> Dict[str, List[int]]:
    assert num_classes % classes_per_task == 0
    n_tasks = num_classes // classes_per_task  # 10
    tasks = {}
    for t in range(n_tasks):
        start = t * classes_per_task
        tasks[f"task{t+1}"] = list(range(start, start + classes_per_task))
    return tasks

# =====tasks=====
tasks = build_tasks(num_classes=200, classes_per_task=20)
task1_classes = tasks["task1"]          # [0..19]
task2_classes = tasks["task2"]          # [20..39]
task3_classes = tasks["task3"]          # [40..59]
task4_classes = tasks["task4"]          # [60..79]
task5_classes = tasks["task5"]          # [80..99]
task6_classes = tasks["task6"]          # [100..119]
task7_classes = tasks["task7"]          # [120..139]
task8_classes = tasks["task8"]          # [140..159]
task9_classes = tasks["task9"]          # [160..179]

class RemapView(Dataset):

    def __init__(self, base_ds: Dataset, indices: List[int], keep_classes: List[int]):
        self.base = base_ds
        self.indices = list(indices)
        keep_sorted = sorted(int(c) for c in keep_classes)
        self.class_to_new = {c:i for i,c in enumerate(keep_sorted)}

    def __len__(self): return len(self.indices)

    def __getitem__(self, i):
        x, y_orig = self.base[self.indices[i]]
        y_new = self.class_to_new[int(y_orig)]
        return x, y_new

def build_remapped_subset(dataset, keep_classes: List[int]) -> RemapView:
    keep_set = set(int(c) for c in keep_classes)
    idx = [i for i, y in enumerate(dataset.targets) if int(y) in keep_set]
    print(f"[DEBUG][Remap] kept={len(idx)} | classes={sorted(keep_set)[:5]}..")
    return RemapView(dataset, idx, keep_classes)

train_09_full = build_remapped_subset(train_set, task9_classes)
test_09       = build_remapped_subset(test_set,  task9_classes)
test_08       = build_remapped_subset(test_set,  task8_classes)
test_07       = build_remapped_subset(test_set,  task7_classes)
test_06       = build_remapped_subset(test_set,  task6_classes)
test_05       = build_remapped_subset(test_set,  task5_classes)
test_04       = build_remapped_subset(test_set,  task4_classes)
test_03       = build_remapped_subset(test_set,  task3_classes)
test_02       = build_remapped_subset(test_set,  task2_classes)
test_01       = build_remapped_subset(test_set,  task1_classes)

def make_loader(ds, bs, shuffle, seed=SEED, num_workers=2):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, generator=g,
                      num_workers=num_workers, pin_memory=(device.type=='cuda'),
                      persistent_workers=(num_workers>0))

# ================== 5)  (ResNet18 + GroupNorm) ==================
def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, 3, stride, 1, bias=False)

def _gn(num_channels: int, num_groups: int = 32):
    return nn.GroupNorm(num_groups, num_channels)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.gn1   = _gn(planes)
        self.conv2 = conv3x3(planes, planes, 1)
        self.gn2   = _gn(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride, bias=False),
                _gn(planes)
            )
    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        out = out + self.shortcut(x)
        return torch.relu(out)

class ResNet18Backbone(nn.Module):
    def __init__(self, nf=64):
        super().__init__()
        self.nf = nf
        self.conv1 = conv3x3(3, nf)
        self.gn1 = _gn(nf)
        self.layer1 = nn.Sequential(BasicBlock(nf, nf, 1), BasicBlock(nf, nf, 1))
        self.layer2 = nn.Sequential(BasicBlock(nf, nf*2, 2), BasicBlock(nf*2, nf*2, 1))
        self.layer3 = nn.Sequential(BasicBlock(nf*2, nf*4, 2), BasicBlock(nf*4, nf*4, 1))
        self.layer4 = nn.Sequential(BasicBlock(nf*4, nf*8, 2), BasicBlock(nf*8, nf*8, 1))
    def forward(self, x):
        x = torch.relu(self.gn1(self.conv1(x)))
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        x = torch.nn.functional.avg_pool2d(x, x.shape[2]); x = x.view(x.size(0), -1)
        return x
    @property
    def out_dim(self): return self.nf*8

class MultiHeadNet(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.heads = nn.ModuleDict()
    def add_head(self, name, num_classes):
        head = nn.Linear(self.backbone.out_dim, num_classes)
        head = head.to(next(self.backbone.parameters()).device)
        self.heads[name] = head
    def forward(self, x, head):
        feat = self.backbone(x)
        return self.heads[head](feat)

# ================== 6) Top-K & Neighbors ==================
with open(topk_path, "rb") as f: topk_CS = pickle.load(f)
with open(neighbors_path, "rb") as f: CS_neighbors = pickle.load(f)
TOPK_COUNT = len(topk_CS)
NEIGH_COUNT = len(CS_neighbors)

# ================== 7) EWC & Freeze Helpers ==================
def build_neighbor_tensors(model, CS_neighbors, neighbor_original_values):
    pm = dict(model.named_parameters())
    usable = [n for n in CS_neighbors if n['name'] in pm]
    if not usable: return {}
    max_f = max((n['cs'] for n in usable), default=1.0) or 1.0
    buckets = {}
    for n in usable:
        buckets.setdefault(n['name'], []).append((int(n['index']), float(n['cs'])/max_f))
    ewc = {}
    for name, lst in buckets.items():
        lst.sort(key=lambda t:t[0])
        idxs = torch.tensor([i for i,_ in lst], device=device, dtype=torch.long)
        fish = torch.tensor([f for _,f in lst], device=device, dtype=torch.float32)
        flat = pm[name].view(-1)
        orig = torch.stack([neighbor_original_values[(name, int(i))] for i in idxs.tolist()]).to(flat.device, dtype=flat.dtype)
        ewc[name] = {'idxs': idxs, 'fish': fish, 'orig': orig}
    return ewc

def build_freeze_masks_and_cache(model, topk_list):
  """
Freeze everything in Top-K **except** the current task head heads.task6.*
(Do not exclude GroupNorm)
"""
    masks, frozen_idxs, frozen_vals = {}, {}, {}
    pm = dict(model.named_parameters())

    by_name = {}
    for e in topk_list:
        n, i = e['name'], int(e['index'])
        if (n in pm) and (not n.startswith("heads.task9.")):
            by_name.setdefault(n, []).append(i)

    for name, idxs in by_name.items():
        p = pm[name]
        flat = p.detach().view(-1)
        idxs_t = torch.tensor(idxs, device=flat.device, dtype=torch.long)

        if p.requires_grad:
            m = torch.ones_like(p, dtype=torch.bool, device=p.device)
            mv = m.view(-1); mv[idxs_t] = False
            masks[name] = mv.view_as(m)

        with torch.no_grad():
            frozen_idxs[name] = idxs_t
            frozen_vals[name] = flat.index_select(0, idxs_t).clone()

    return masks, frozen_idxs, frozen_vals

def apply_freeze_after_backward(model, masks):
    with torch.no_grad():
        for n,p in model.named_parameters():
            m = masks.get(n, None)
            if p.grad is not None and m is not None:
                p.grad.mul_(m.to(p.grad.dtype))

@torch.no_grad()
def apply_strict_freeze_after_step(param_map, frozen_idxs, frozen_vals):
    for n, idxs in frozen_idxs.items():
        if n in param_map:
            flat = param_map[n].view(-1)
            flat.index_copy_(0, idxs, frozen_vals[n].to(flat.device, dtype=flat.dtype))

def mask_stats(masks):
    total = sum(m.numel() for m in masks.values())
    frozen = sum((~m).sum().item() for m in masks.values())
    return total, frozen

def audit_topk_vs_masks(model, topk_list):
    pm = dict(model.named_parameters())
    unique_pairs = set((e['name'], int(e['index'])) for e in topk_list)
    excl_not_found = excl_head_t9 = excl_no_grad = excl_oob = 0
    included = set()
    for name, idx in unique_pairs:
        p = pm.get(name, None)
        if p is None: excl_not_found += 1; continue
        if name.startswith("heads.task9."): excl_head_t9 += 1; continue
        if idx < 0 or idx >= p.numel(): excl_oob += 1; continue
        if not p.requires_grad: excl_no_grad += 1
        included.add((name, idx))
    print(f"[AUDIT] TopK unique pairs     : {len(unique_pairs)}")
    print(f"[AUDIT] Excluded heads.task9.*: {excl_head_t9}")
    print(f"[AUDIT] Not found             : {excl_not_found}")
    print(f"[AUDIT] Out-of-bounds         : {excl_oob}")
    print(f"[AUDIT] No-grad params        : {excl_no_grad}")
    print(f"[AUDIT] Will be masked/strict : {len(included)}")

def freeze_backbone_bn_running_stats(model):
    for m in model.backbone.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

# ================== 8) Initialize model and regularizers ==================
def init_model_and_regularizers(lambda_ewc):
    backbone = ResNet18Backbone(nf=64).to(device)
    model = MultiHeadNet(backbone).to(device)
    for t in range(1, 10+1):
        model.add_head(f"task{t}", 20)
    model.to(device)

    ckpt = torch.load(model_path, map_location=device)
    sd = ckpt["model_state"] if "model_state" in ckpt else ckpt
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing:    print("[INIT] Missing keys:", missing)
    if unexpected: print("[INIT] Unexpected keys:", unexpected)

    # (1) Fully freeze task1..task9 heads
    for name in [f"task{i}" for i in range(1, 9)]:
        for p in model.heads[name].parameters():
            p.requires_grad = False
    print("[INIT] Frozen heads: task1..task8")

    # (2) Copy task2 head weights -> task10 head (warm-start)
    with torch.no_grad():
        if ("task2" in model.heads) and ("task9" in model.heads):
            h2 = model.heads["task2"]; h9 = model.heads["task9"]
            same_W = (h2.weight.shape == h9.weight.shape)
            same_b = (h2.bias is not None) and (h9.bias is not None) and (h2.bias.shape == h9.bias.shape)
            if same_W: h9.weight.copy_(h2.weight)
            if same_b: h9.bias.copy_(h2.bias)
            print(f"[INIT] Copied task2 → task9 head | W={same_W} | b={same_b}")
        else:
            print("[INIT] WARN: task2/task9 head not found — skip head weight copy")

    # (3) Prepare EWC: save reference values
    with torch.no_grad():
        cpu_cache = {n: p.view(-1).detach().cpu() for n,p in model.named_parameters()}

    neighbor_original_values = {}
    for n in CS_neighbors:
        name, idx = n['name'], int(n['index'])
        if name in cpu_cache and idx < cpu_cache[name].numel():
            neighbor_original_values[(name, idx)] = cpu_cache[name][idx]

    ewc_tensors = build_neighbor_tensors(model, CS_neighbors, neighbor_original_values)

    # (4) Top-K freeze masks
    masks, frozen_idxs, frozen_vals = build_freeze_masks_and_cache(model, topk_CS)

    audit_topk_vs_masks(model, topk_CS)
    return model, ewc_tensors, masks, frozen_idxs, frozen_vals

# ================== 9) evalution ==================
@torch.no_grad()
def evaluate_head(model, loader, head):
    model.eval(); correct=0; total=0
    for x,y in loader:
        x = x.to(device); y = torch.as_tensor(y, device=device, dtype=torch.long)
        logits = model(x, head=head); pred = logits.argmax(1)
        correct += (pred==y).sum().item(); total += y.size(0)
    return 100.0*correct/max(1,total)

def train_one_setting(lr_backbone, lr_head, bs, lambda_ewc, epochs):
    model, ewc_tensors, masks, frozen_idxs, frozen_vals = init_model_and_regularizers(lambda_ewc)

    bb_decay, bb_nodecay, hd_decay, hd_nodecay = [], [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        is_head9 = n.startswith("heads.task9.")
        no_decay = (p.dim()==1) or n.endswith(".bias") or ("bn" in n.lower()) or ("gn" in n.lower())
        if is_head9:
            (hd_nodecay if no_decay else hd_decay).append(p)
        else:
            (bb_nodecay if no_decay else bb_decay).append(p)

    optimizer = optim.SGD(
        [
            {"params": bb_decay,   "lr": lr_backbone, "weight_decay": 5e-4},
            {"params": bb_nodecay, "lr": lr_backbone, "weight_decay": 0.0},
            {"params": hd_decay,   "lr": lr_head,     "weight_decay": 1e-4},
            {"params": hd_nodecay, "lr": lr_head,     "weight_decay": 0.0},
        ],
        momentum=0.9
    )

    param_map = {n:p for n,p in model.named_parameters()}

    train_loader = make_loader(train_09_full, bs=bs, shuffle=True)
    test1_loader = make_loader(test_01,       bs=256, shuffle=False)
    test2_loader = make_loader(test_02,       bs=256, shuffle=False)
    test3_loader = make_loader(test_03,       bs=256, shuffle=False)
    test4_loader = make_loader(test_04,       bs=256, shuffle=False)
    test5_loader = make_loader(test_05,       bs=256, shuffle=False)
    test6_loader = make_loader(test_06,       bs=256, shuffle=False)
    test7_loader = make_loader(test_07,       bs=256, shuffle=False)
    test8_loader = make_loader(test_08,       bs=256, shuffle=False)
    test9_loader = make_loader(test_09,       bs=256, shuffle=False)

    pre = [
        evaluate_head(model, test1_loader, head="task1"),
        evaluate_head(model, test2_loader, head="task2"),
        evaluate_head(model, test3_loader, head="task3"),
        evaluate_head(model, test4_loader, head="task4"),
        evaluate_head(model, test5_loader, head="task5"),
        evaluate_head(model, test6_loader, head="task6"),
        evaluate_head(model, test7_loader, head="task7"),
        evaluate_head(model, test8_loader, head="task8"),
    ]
    print(f"[PRE] BEFORE FT Task9 | " + " ".join([f"T{i+1}={pre[i]:.2f}%" for i in range(8)]))

    total_mask_elems, total_frozen = mask_stats(masks)
    print(f"[INFO]   mask_elems={total_mask_elems} | frozen(TopK)={total_frozen}")
    print(f"[OPT ]   lr_backbone={lr_backbone} | lr_head={lr_head} | "
          f"bb_decay={len(bb_decay)} bb_nodecay={len(bb_nodecay)} | "
          f"hd_decay={len(hd_decay)} hd_nodecay={len(hd_nodecay)}")

    best = {'epoch': -1, 't1': -1.0, 't2': -1.0, 't3': -1.0, 't4': -1.0, 't5': -1.0, 't6': -1.0, 't7': -1.0, 't8': -1.0, 't9': -1.0, 'avg': -1.0, 'model_state': None}
    for e in range(1, epochs+1):
        model.train()
        freeze_backbone_bn_running_stats(model)

        running_loss=0.0; steps=0
        for imgs, labels in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = torch.as_tensor(labels, device=device, dtype=torch.long)
            optimizer.zero_grad(set_to_none=True)

            if (labels.min() < 0) or (labels.max() >= 20):
                bad = labels[(labels < 0) | (labels >= 20)]
                raise RuntimeError(f"[DEBUG] Task9 expects labels in 0..19, found range "
                                   f"{int(labels.min())}..{int(labels.max())} | sample={bad[:10].tolist()}")

            # EWC penalty
            ewc_penalty = 0.0
            if lambda_ewc != 0 and len(ewc_tensors) > 0:
                for name, pack in ewc_tensors.items():
                    p = param_map[name].view(-1)
                    diff = p.index_select(0, pack['idxs']) - pack['orig']
                    ewc_penalty += (pack['fish'] * (diff**2)).sum()

            logits = model(imgs, head="task9")
            if logits.shape[1] != 20:
                raise RuntimeError(f"[DEBUG] logits.shape[1]={logits.shape[1]} != 20 (head/task mismatch)")

            loss = nn.functional.cross_entropy(logits, labels) + (lambda_ewc/2.0)*ewc_penalty

            loss.backward()
            apply_freeze_after_backward(model, masks)
            optimizer.step()
            apply_strict_freeze_after_step(param_map, frozen_idxs, frozen_vals)

            running_loss += float(loss.detach().cpu()); steps += 1

        acc_t1 = evaluate_head(model, test1_loader, head="task1")
        acc_t2 = evaluate_head(model, test2_loader, head="task2")
        acc_t3 = evaluate_head(model, test3_loader, head="task3")
        acc_t4 = evaluate_head(model, test4_loader, head="task4")
        acc_t5 = evaluate_head(model, test5_loader, head="task5")
        acc_t6 = evaluate_head(model, test6_loader, head="task6")
        acc_t7 = evaluate_head(model, test7_loader, head="task7")
        acc_t8 = evaluate_head(model, test8_loader, head="task8")
        acc_t9 = evaluate_head(model, test9_loader, head="task9")
        avg = (acc_t1 + acc_t2 + acc_t3 + acc_t4 + acc_t5 + acc_t6 + acc_t7 + acc_t8 + acc_t9)/9.0

        print(f"  Epoch {e}/{epochs} | train_loss={running_loss/max(1,steps):.4f} | "
              f"T1={acc_t1:.2f}% T2={acc_t2:.2f}% T3={acc_t3:.2f}% T4={acc_t4:.2f}% "
              f"T5={acc_t5:.2f}% T6={acc_t6:.2f}% T7={acc_t7:.2f}% T8={acc_t8:.2f}% T9={acc_t9:.2f}% AVG9={avg:.2f}%")

        if (avg > best['avg']) or (avg == best['avg'] and acc_t9 > best['t9']):
            best = {'epoch': e, 't1': acc_t1, 't2': acc_t2, 't3': acc_t3, 't4': acc_t4,
                    't5': acc_t5, 't6': acc_t6, 't7': acc_t7, 't8': acc_t8, 't9': acc_t9, 'avg': avg,
                    'model_state': {k:v.detach().cpu() for k,v in model.state_dict().items()}}

    return best

# ================== 11) GRID SEARCH  ==================
def run_grid_search_and_save_best():
    backbone_lrs = [1e-6]
    head_lrs     = [0.7]
    batch_sizes  = [32]
    lambda_values  = [2]
    epoch_counts   = [150]

    print(f"[DATA ] Train(T9)={len(train_09_full)} | Test(T9)={len(test_09)} | Test(T8)={len(test_08)} | Test(T7)={len(test_07)} | Test(T6)={len(test_06)} | Test(T5)={len(test_05)} | Test(T4)={len(test_04)} | Test(T3)={len(test_03)} | Test(T2)={len(test_02)} | Test(T1)={len(test_01)}")
    print(f"[META ] TopK={TOPK_COUNT} | Neighbors={NEIGH_COUNT}")

    global_best = {'avg': -1.0, 't1': -1.0, 't2': -1.0, 't3': -1.0, 't4': -1.0, 't5': -1.0, 't6': -1.0, 't7': -1.0, 't8': -1.0, 't9': -1.0,
                   'epoch': -1, 'setting': None, 'state': None}

    for lr_bb, lr_hd, bs, lam, ep in itertools.product(backbone_lrs, head_lrs, batch_sizes, lambda_values, epoch_counts):
        print(f"\n[SETTING] LR_backbone={lr_bb}, LR_head={lr_hd}, BS={bs}, λ={lam}, EPOCHS={ep} "
              f"| TopK={TOPK_COUNT}, Neigh={NEIGH_COUNT}")
        best = train_one_setting(lr_backbone=lr_bb, lr_head=lr_hd, bs=bs, lambda_ewc=lam, epochs=ep)
        print(f"[SETTING-BEST] epoch {best['epoch']} | "
              f"T1={best['t1']:.2f}% | T2={best['t2']:.2f}% | T3={best['t3']:.2f}% | T4={best['t4']:.2f}% | "
              f"T5={best['t5']:.2f}% | T6={best['t6']:.2f}% | T7={best['t7']:.2f}% | T8={best['t8']:.2f}% | T9={best['t9']:.2f}% | AVG9={best['avg']:.2f}%")
        if best['avg'] > global_best['avg']:
            global_best = {
                'avg': best['avg'], 't1': best['t1'], 't2': best['t2'], 't3': best['t3'], 't4': best['t4'],
                't5': best['t5'], 't6': best['t6'], 't7': best['t7'], 't8': best['t8'], 't9': best['t9'],
                'epoch': best['epoch'],
                'setting': f"LR_backbone={lr_bb}, LR_head={lr_hd}, BS={bs}, λ={lam}, EPOCHS={ep}",
                'state': best['model_state']
            }

    best_test_path = os.path.join(ckpt_dir, "Gtask9_best_test_for_finetune_ResNet18_GN_CS.pth")
    torch.save({
        "model_state": global_best['state'],
        "best_config": global_best['setting'],
        "best_test_acc": global_best['t9'],
        "best_test_epoch": global_best['epoch'],
        "trained_head": "task9",
        "all_heads": [f"task{i}" for i in range(1, 11)],
        "acc_t1": global_best['t1'],
        "acc_t2": global_best['t2'],
        "acc_t3": global_best['t3'],
        "acc_t4": global_best['t4'],
        "acc_t5": global_best['t5'],
        "acc_t6": global_best['t6'],
        "acc_t7": global_best['t7'],
        "acc_t8": global_best['t8'],
        "acc_t9": global_best['t9'],
        "acc_avg": global_best['avg'],
        "saved_at": datetime.now().isoformat(),
        "arch": "ResNet18-GN-nf64",
    }, best_test_path)

    print("\n====================")
    print(f"[GLOBAL BEST] AVG9={global_best['avg']:.2f}% | "
          f"T1={global_best['t1']:.2f}% | T2={global_best['t2']:.2f}% | "
          f"T3={global_best['t3']:.2f}% | T4={global_best['t4']:.2f}% | T5={global_best['t5']:.2f}% | "
          f"T6={global_best['t6']:.2f}% | T7={global_best['t7']:.2f}% | T8={global_best['t8']:.2f}% | T9={global_best['t9']:.2f}% | "
          f"at epoch {global_best['epoch']}")
    print(f"[SETTING     ] {global_best['setting']}")
    print(f"[SAVED       ] {best_test_path}")

# ================== 12) run ==================
run_grid_search_and_save_best()


Mounted at /content/drive
[INFO] Started at 2026-09-13T10:03:14.581034
[INFO] torch.cuda.is_available() -> True
[INFO] CUDA sanity OK on device: cuda:0
[INFO] Using device: cuda
[INFO] CUDA device: NVIDIA RTX PRO 6000 Blackwell Server Edition
[INFO] CUDA capability: (12, 0)
[INFO] CUDA current mem (MB): 2.0
[INFO] Seeds set to 42
[OK] Checkpoint(prev) exists (43.02 MB): /content/drive/MyDrive/ML_Project/project_files/Group_norm/Gtask8_best_test_for_finetune_ResNet18_GN_CS.pth
[OK] CS Top-K exists (65.88 MB): /content/drive/MyDrive/ML_Project/project_files/Group_norm/groupNorm_CS_task1_2_3_4_5_6_7_8_tiny_imageNet_topk.pkl
[OK] CS Neighbors exists (32.47 MB): /content/drive/MyDrive/ML_Project/project_files/Group_norm/groupNorm_CS_task1_2_3_4_5_6_7_8_tiny_imageNet_neighbors.pkl
[INFO] Found processed data at: /content/drive/MyDrive/ML_Project/data/TINYIMG/processed
[INFO] Loading Tiny-ImageNet ...
[INFO] Train size=100000 | Val/Test size=10000
[DEBUG][Remap] kept=10000 | classes=[160, 161